[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/02_mathematics_for_ml/03_rank_inverse_eigen_svd.ipynb)

# Bài 03 — Độc lập tuyến tính, hạng, nghịch đảo, trị riêng và SVD

**🎯 Mục tiêu**
- Hiểu độc lập tuyến tính và hạng (rank) → giải thích **multicollinearity** trong Linear Regression.
- Giải hệ phương trình tuyến tính đúng cách; hiểu **condition number** và vì sao không nên dùng `inv`.
- Eigenvalue/eigenvector: ý nghĩa hình học, power iteration from scratch → nền tảng của **PCA**.
- SVD và xấp xỉ hạng thấp → nén ảnh, PCA, **recommender system**.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/02_mathematics_for_ml

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Độc lập tuyến tính và hạng

Các vector $\mathbf v_1, \dots, \mathbf v_k$ **độc lập tuyến tính** nếu không vector nào biểu diễn được bằng tổ hợp tuyến tính của các vector còn lại. **Hạng** (rank) của ma trận = số cột (hoặc hàng) độc lập tuyến tính tối đa.

Trong ML, các **cột** của $X$ là các feature. Nếu một feature là tổ hợp tuyến tính của các feature khác (vd: `total_price = area × price_per_m2` sau log, hay có cả cột `°C` và `°F`), thì $X$ **không có hạng đầy đủ** → $X^\top X$ không khả nghịch → hệ số Linear Regression không xác định duy nhất.

In [ ]:
area = rng.uniform(30, 120, 50)
rooms = rng.integers(1, 5, 50)
X_ok = np.column_stack([area, rooms])
X_bad = np.column_stack([area, rooms, 2 * area + 3 * rooms])   # cột 3 = tổ hợp của cột 1, 2

print("rank X_ok :", np.linalg.matrix_rank(X_ok), "/", X_ok.shape[1])
print("rank X_bad:", np.linalg.matrix_rank(X_bad), "/", X_bad.shape[1], "→ thiếu hạng!")

## 2. Ma trận nghịch đảo và giải hệ phương trình

Ma trận vuông $A$ khả nghịch nếu tồn tại $A^{-1}$ với $AA^{-1} = I$. Điều kiện: $\det(A) \ne 0$ ⇔ $A$ có hạng đầy đủ.

Hệ $A\mathbf x = \mathbf b$ có nghiệm $\mathbf x = A^{-1}\mathbf b$. **Nhưng trong code, đừng tính `inv(A) @ b`** — dùng `np.linalg.solve` (nhanh hơn, chính xác hơn) hoặc `np.linalg.lstsq` (khi hệ không vuông / thiếu hạng).

In [ ]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
b = np.array([9.0, 8.0])
x_solve = np.linalg.solve(A, b)
x_inv = np.linalg.inv(A) @ b
print("solve:", x_solve, "| inv:", x_inv, "| det(A) =", np.linalg.det(A))
print("Kiểm tra A @ x == b:", np.allclose(A @ x_solve, b))

### Condition number — khi bài toán "gần như" không giải được

**Condition number** $\kappa(A) = \sigma_{max}/\sigma_{min}$ đo độ nhạy của nghiệm với sai số nhỏ của dữ liệu. $\kappa$ lớn (vd > $10^{10}$) → kết quả không đáng tin. Đây là điều xảy ra khi hai feature **gần như** trùng nhau (multicollinearity mạnh).

In [ ]:
for eps in [1.0, 1e-3, 1e-8]:
    A = np.array([[1.0, 1.0], [1.0, 1.0 + eps]])     # hai hàng gần như trùng nhau khi eps nhỏ
    b = np.array([2.0, 2.0 + eps])
    b_noisy = b + np.array([0, 1e-6])                # nhiễu rất nhỏ ở dữ liệu
    x1, x2 = np.linalg.solve(A, b), np.linalg.solve(A, b_noisy)
    print(f"eps={eps:<7g} cond={np.linalg.cond(A):10.2e}  nghiệm={x1.round(4)}  nghiệm khi có nhiễu={x2.round(4)}")

Với `eps = 1e-8`, nhiễu $10^{-6}$ làm nghiệm thay đổi hàng trăm đơn vị! Ở chương 06, ta sẽ thấy **Ridge regression** sửa vấn đề này bằng cách giải $(X^\top X + \lambda I)\mathbf w = X^\top\mathbf y$ — cộng $\lambda I$ làm ma trận "ổn định" hơn.

## 3. Trị riêng và vector riêng (eigenvalue / eigenvector)

Vector $\mathbf v \ne 0$ là **vector riêng** của ma trận vuông $A$ nếu $A$ chỉ **co giãn** nó (không đổi hướng):

$$A\mathbf v = \lambda\mathbf v$$

$\lambda$ là **trị riêng** tương ứng. Với ma trận đối xứng (như ma trận hiệp phương sai), các vector riêng **vuông góc** với nhau và trị riêng là số thực.

In [ ]:
A = np.array([[3.0, 1.0], [1.0, 2.0]])
eigvals, eigvecs = np.linalg.eigh(A)          # eigh cho ma trận đối xứng (ổn định hơn eig)
print("trị riêng:", eigvals.round(4))
for lam, v in zip(eigvals, eigvecs.T):
    print(f"A v = {(A @ v).round(4)}  =  λ v = {(lam * v).round(4)}")

**Ý nghĩa hình học:** biến đổi vòng tròn đơn vị bằng $A$ được một hình elip; các vector riêng là **trục** của elip, trị riêng là **độ dài** các bán trục.

In [ ]:
t = np.linspace(0, 2 * np.pi, 200)
circle = np.c_[np.cos(t), np.sin(t)]
ellipse = circle @ A.T
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.plot(circle[:, 0], circle[:, 1], "--", label="vòng tròn đơn vị")
ax.plot(ellipse[:, 0], ellipse[:, 1], label="sau khi nhân A")
for lam, v in zip(eigvals, eigvecs.T):
    ax.arrow(0, 0, *(lam * v), width=0.03, color="red", length_includes_head=True)
ax.set_aspect("equal")
ax.legend(loc="lower right")
ax.set_title("Vector riêng = trục của elip, trị riêng = độ dài")
plt.show()

### Power iteration — tìm vector riêng lớn nhất from scratch

Nhân lặp lại một vector ngẫu nhiên với $A$ rồi chuẩn hoá: thành phần theo vector riêng có $|\lambda|$ lớn nhất sẽ "thắng" dần. Đây là ý tưởng của PageRank và một số thuật toán PCA quy mô lớn.

In [ ]:
def power_iteration(A, n_iters=100, seed=0):
    v = np.random.default_rng(seed).normal(size=A.shape[0])
    for _ in range(n_iters):
        v = A @ v
        v /= np.linalg.norm(v)
    return v @ A @ v, v      # Rayleigh quotient = trị riêng tương ứng


lam, v = power_iteration(A)
print("power iteration:", round(lam, 6), v.round(4))
print("numpy eigh      :", eigvals.max().round(6), eigvecs[:, np.argmax(eigvals)].round(4), "(có thể ngược dấu)")

### Liên hệ PCA (chương 21)

Vector riêng của **ma trận hiệp phương sai** của dữ liệu chỉ ra các hướng dữ liệu biến thiên nhiều nhất; trị riêng = phương sai theo hướng đó. PCA chiếu dữ liệu lên các vector riêng có trị riêng lớn nhất.

In [ ]:
cov_true = np.array([[3.0, 2.0], [2.0, 2.0]])
data = rng.multivariate_normal([0, 0], cov_true, size=500)
C = np.cov(data.T)
vals, vecs = np.linalg.eigh(C)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(data[:, 0], data[:, 1], s=8, alpha=0.4)
for val, vec in zip(vals, vecs.T):
    ax.arrow(0, 0, *(2 * np.sqrt(val) * vec), width=0.06, color="red", length_includes_head=True)
ax.set_aspect("equal")
ax.set_title("Vector riêng của ma trận hiệp phương sai = hướng biến thiên chính")
plt.show()
print("phương sai theo các hướng chính:", vals.round(3))

## 4. SVD — Singular Value Decomposition

**Mọi** ma trận $X$ ($n \times d$) đều phân tích được thành:

$$X = U\Sigma V^\top$$

- $U$ ($n \times r$), $V$ ($d \times r$) có các cột trực chuẩn; $\Sigma$ là ma trận đường chéo các **giá trị kỳ dị** $\sigma_1 \ge \sigma_2 \ge \dots \ge 0$.
- Liên hệ với eigen: $X^\top X = V\Sigma^2V^\top$ → giá trị kỳ dị $= \sqrt{\text{trị riêng của } X^\top X}$.

**Xấp xỉ hạng thấp (Eckart–Young):** giữ $k$ giá trị kỳ dị lớn nhất, $X_k = U_k\Sigma_kV_k^\top$ là xấp xỉ hạng $k$ **tốt nhất** của $X$. Ứng dụng: nén dữ liệu, khử nhiễu, PCA, recommender (ma trận user × item).

In [ ]:
from sklearn.datasets import load_sample_image

img = load_sample_image("china.jpg").mean(axis=2) / 255.0      # ảnh xám có sẵn trong sklearn
U, S, Vt = np.linalg.svd(img, full_matrices=False)
print("ảnh:", img.shape, "| số giá trị kỳ dị:", len(S))

fig, axes = plt.subplots(1, 4, figsize=(16, 3.8))
for ax, k in zip(axes, [5, 20, 50, len(S)]):
    approx = (U[:, :k] * S[:k]) @ Vt[:k]
    stored = k * (U.shape[0] + Vt.shape[1] + 1)
    ax.imshow(approx, cmap="gray")
    ax.set_title(f"k = {k}  ({stored / img.size:.0%} dữ liệu)")
    ax.axis("off")
plt.show()

In [ ]:
energy = np.cumsum(S**2) / np.sum(S**2)
fig, ax = plt.subplots()
ax.plot(np.arange(1, len(S) + 1), energy)
ax.axhline(0.95, ls="--", color="gray")
ax.set_xscale("log")
ax.set_xlabel("số giá trị kỳ dị giữ lại (k)")
ax.set_ylabel("tỉ lệ 'năng lượng' giữ lại")
ax.set_title("Phần lớn thông tin nằm trong vài giá trị kỳ dị đầu tiên")
plt.show()
print("k cần để giữ 95% năng lượng:", int(np.searchsorted(energy, 0.95) + 1))

## 5. Tổng kết liên hệ ML

```text
rank / độc lập tuyến tính  → multicollinearity, feature trùng lặp (ch.06, 18)
solve / lstsq              → normal equation của Linear Regression (ch.06)
condition number           → vì sao cần scaling & Ridge (ch.04, 06)
eigenvector                → PCA (ch.21), spectral clustering, PageRank
SVD                        → PCA, nén ảnh, recommender (matrix factorization, ch.26), LSA trong NLP
```

## 🧠 Kiểm tra nhanh

1. $X$ có 5 cột nhưng rank = 4. Điều này nói gì về các feature?
2. Vì sao nên dùng `np.linalg.solve(A, b)` thay vì `np.linalg.inv(A) @ b`?
3. Ma trận hiệp phương sai có trị riêng $[0.01, 4.5]$. Dữ liệu "trải" chủ yếu theo mấy chiều?
4. Lưu xấp xỉ hạng $k$ của ảnh $m\times n$ cần bao nhiêu số?

## 📌 Tóm tắt

- Rank < số cột ⇔ có feature dư thừa tuyến tính ⇔ $X^\top X$ suy biến.
- Condition number lớn → nghiệm cực nhạy với nhiễu → cần regularization / bỏ feature.
- $A\mathbf v = \lambda\mathbf v$: vector riêng là hướng bị co giãn thuần tuý; với ma trận hiệp phương sai, đó là hướng biến thiên chính.
- SVD tồn tại cho mọi ma trận; xấp xỉ hạng thấp tốt nhất = giữ $k$ giá trị kỳ dị lớn nhất.

## 📚 Tài liệu tham khảo

- 3Blue1Brown — *Essence of Linear Algebra*, chương 7 (inverse), 14 (eigen)
- Gilbert Strang — *Linear Algebra and Learning from Data* (2019)
- Vũ Hữu Tiệp — [Machine Learning cơ bản, phần toán](https://machinelearningcoban.com/math/)